# Open-Meteo Weather Data Exploration  

## Goals  

Using **Hailey's ecoregions centroid coordinates** (1500+ points), we aim to develop a system that identifies:  

- **Top 100 hottest locations**  
- **Top 100 coldest locations**  
- **Top 100 wettest locations**  

These rankings will be based on **forecasted** temperature (`temperature_2m_max`, `temperature_2m_min`) and precipitation (`precipitation_sum`) for the **next day**. However, the code is also capable of using **historical data**, allowing for flexible analysis (I tested code for historical data available on openmeteo for all 3 metrics and it ran smoothly and was consistent with real world findings of hottest/coldest places).  

### Why Forecasted Data?  

We opted for forecasted data to create a **dynamic system** where Pokémon “move” daily based on changing weather conditions.  

For example, in our system:  

- **Ho-Oh** (the strongest Fire-type Pokémon) will reside in the **hottest biome centroid** for the day.  
  - If today's forecast predicts **Gobi Desert** as the hottest location, Ho-Oh will be there.  
  - If tomorrow's forecast shifts the hottest location to **Grand Canyon**, Ho-Oh will move accordingly.  

This logic applies to all other Pokémon, ranked by strength and assigned based on the **hottest, coldest, and wettest places** globally.  

### Implementation  

- **Data Collection**: Open-Meteo was queried to retrieve the required weather data.  
- **Database Formation**: The results were compiled into a structured database, ranking locations based on temperature and precipitation.  
- **Pokémon Matching**:  
  - **Adrian** will match the ranked locations with Pokémon.  
  - This is based on [**Jonathan Chang’s Pokémon database**](../../data/main.db), ensuring each Pokémon is placed in a thematically appropriate biome.  

This approach allows for an interactive and ever-changing Pokémon-world simulation driven by real-world weather data.  

---



In [1]:
import concurrent.futures
import nest_asyncio
import asyncio
import aiohttp
import pandas as pd
import json
import requests
from datetime import datetime, timedelta
import concurrent.futures
import time

# **Asynchronous Weather Data Fetching Script Explanation**  

## **Overview**  
This script **fetches weather forecast data** (temperature and rainfall) asynchronously from the Open-Meteo API for multiple ecoregions. It then processes the data and saves the **hottest, coldest, and wettest places** to CSV files.  

The script uses **`asyncio` and `aiohttp`** for efficient parallel execution, significantly improving speed compared to traditional synchronous requests.  

---

### **1.Enabling Nested Async (For Jupyter)**  
```python
nest_asyncio.apply()
```
- This ensures **Jupyter Notebook** can run `asyncio` properly by preventing event loop conflicts.  

---

### **2.Flattening the JSON Data**  
```python
coordinates = [
    {"region": region_name, "latitude": region_data["centroid"][0], "longitude": region_data["centroid"][1]}
    for region_type, regions in data.items()
    for region_name, region_data in regions.items()
]
```
- Converts the JSON structure into a **list of dictionaries** for easier processing (using .items() function that return dict_item datatype).  
- Each dictionary contains:
  - `"region"` → Name of the ecoregion.  
  - `"latitude"` and `"longitude"` → Geographical coordinates.  

---

### **3.Fetching Weather Forecast Data Asynchronously**  
- **Asynchronous function** that fetches **daily max & min temperature and rainfall** for a given location.  
- Uses **Open-Meteo API**, requesting **daily temperature and precipitation data**.  

### **4.Handling API Rate Limits and Errors**  
- Uses **`session.get()`** for an asynchronous API request.  
- **Rate limit handling**:  
  - If **HTTP 429 (Too Many Requests)** is returned, it **waits (`await asyncio.sleep()`)** before retrying.  
  - Retries up to **3 times** before giving up.   

### **5.Processing the API Response**  
- Extracts **temperature and rainfall data** from the API response.  
- Returns a dictionary containing:  
  - `"region"` → Name of the ecoregion.  
  - `"max_temperature"` → Maximum daily temperature.  
  - `"min_temperature"` → Minimum daily temperature.  
  - `"max_rainfall"` → Maximum daily precipitation.  

### **6.Handling Request Errors**  
```python
        except Exception as e:
            print(f"Error for {coord['region']}: {e}")
    return None
```
- Logs any **network or API failures**.  
- Returns `None` if no valid data is retrieved after **3 retries**.  

---

## Processing All Coordinates Asynchronously**  

- **Creates an `aiohttp.ClientSession()`** for managing multiple API requests efficiently.  
- Uses **list comprehension** to create a list of tasks (`fetch_forecast_data(session, coord)`).  
- `asyncio.gather(*tasks, return_exceptions=True)` → **Runs all requests in parallel**.  
- Filters out `None` values (failed requests).  

---

## ** Main Function: Sorting and Saving Results**  
- Calls `process_all_coordinates()` to **fetch weather data** asynchronously.  
- Converts the results into a **pandas DataFrame**.  

## **Key Features of the Script**  
✅ Uses **asynchronous requests** for faster execution.  
✅ Implements **rate limit handling** to prevent API failures.  
✅ Sorts and **extracts the hottest, coldest, and wettest places**.  



In [2]:
import nest_asyncio
import asyncio
import aiohttp
import pandas as pd
import json

nest_asyncio.apply()

# Load JSON file
file_path = "../../data/biomes_data/Ecoregions_Coordinates.json"
with open(file_path, "r") as f:
    data = json.load(f)

# Flatten data
# .items function returns a sort of list (dict_items data type that is non viewable that gives a iterable but not printable list of key-value pairs as tuples, so at top level I get biome and associated dictionaries) 
coordinates = [
    {"region": region_name, "latitude": region_data["centroid"][0], "longitude": region_data["centroid"][1]}
    for region_type, regions in data.items()
    for region_name, region_data in regions.items()
]

# Fetch data for temperature and rainfall
async def fetch_forecast_data(session, coord):
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": coord["latitude"],
        "longitude": coord["longitude"],
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "timezone": "UTC",
    }

    retries = 3
    for attempt in range(retries):
        try:
            async with session.get(BASE_URL, params=params, timeout=15) as response:
                if response.status == 429:
                    retry_after = int(response.headers.get("Retry-After", 5))
                    await asyncio.sleep(retry_after)
                    continue
                response.raise_for_status()
                data = await response.json()
                if "daily" in data:
                    return {
                        "region": coord["region"],
                        "max_temperature": data["daily"]["temperature_2m_max"][0],
                        "min_temperature": data["daily"]["temperature_2m_min"][0],
                        "max_rainfall": data["daily"]["precipitation_sum"][0],
                    }
        except Exception as e:
            print(f"Error for {coord['region']}: {e}")
    return None

# Process all coordinates
async def process_all_coordinates():
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_forecast_data(session, coord) for coord in coordinates]
        results = await asyncio.gather(*tasks, return_exceptions=True)
    return [res for res in results if res]

# Main function
async def main():
    results = await process_all_coordinates()
    df = pd.DataFrame(results)

    hottest = df.sort_values(by="max_temperature", ascending=False).head(100)
    coldest = df.sort_values(by="min_temperature").head(100)
    wettest = df.sort_values(by="max_rainfall", ascending=False).head(100)

    hottest.to_csv("../../data/weather_data/hottest_places.csv", index=False)
    coldest.to_csv("../../data/weather_data/coldest_places.csv", index=False)
    wettest.to_csv("../../data/weather_data/wettest_places.csv", index=False)

# Run the async function
await main()
